# Parameter experiments

Create unregistered strategy instances with different parameters, then compare them through the same tested backtest engine.


In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from retail_sp500.backtest import BacktestConfig, run_many
from retail_sp500.data import load_shiller_data, synthetic_market_data
from retail_sp500.plotting import comparison_figure
from retail_sp500.strategies import MovingAverageTrend, TrendVolatilityTarget, VolatilityTarget

pd.set_option("display.max_columns", 100)


In [ ]:
USE_SYNTHETIC = True
market = synthetic_market_data(periods=600) if USE_SYNTHETIC else load_shiller_data()

config = BacktestConfig(
    initial_cash=100_000,
    monthly_contribution=1_000,
    cash_annual_return=0.0,
)


## Define a small, interpretable experiment set

Avoid broad parameter searches. A compact grid makes overfitting easier to see and explain.


In [ ]:
experimental_strategies = [
    MovingAverageTrend(
        months=10,
        buffer=0.01,
        name="Trend: 10 months, 1% buffer",
    ),
    MovingAverageTrend(
        months=12,
        buffer=0.02,
        name="Trend: 12 months, 2% buffer",
    ),
    VolatilityTarget(
        target_volatility=0.10,
        lookback_months=12,
        name="Volatility target: 10%",
    ),
    VolatilityTarget(
        target_volatility=0.15,
        lookback_months=12,
        name="Volatility target: 15%",
    ),
    TrendVolatilityTarget(
        trend_months=10,
        trend_buffer=0.01,
        target_volatility=0.12,
        name="Trend 10m + volatility 12%",
    ),
]

results = run_many(market, experimental_strategies, config)


In [ ]:
metrics = pd.DataFrame(
    {name: result.metrics for name, result in results.items()}
).T

metrics[[
    "ending_value",
    "time_weighted_cagr",
    "annualized_volatility",
    "max_drawdown",
    "sharpe_zero_cash",
    "average_etf_weight",
]].sort_values("ending_value", ascending=False)


In [ ]:
comparison_figure(results).show()


## Inspect allocations


In [ ]:
allocation_history = pd.DataFrame(
    {name: result.history["target_weight"] for name, result in results.items()}
)

allocation_history.tail(36)
